# UD5.05. Curvas, regularización y retrollamadas

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloques 10, 11 y 12 de los apuntes · Criterios **2.b**, **2.d** y **2.e**

---

En la UD4 se evaluaba un modelo ya entrenado. Aquí se evalúa **el entrenamiento mismo**,
que es un objeto distinto y se mira antes: si las curvas dicen que el modelo memoriza, las
métricas finales ya no hacen falta para saber que hay trabajo pendiente.

Tres cosas:

1. **Fabricar los cinco patrones a propósito** y aprender a reconocerlos. Un modelo que
   falla de las cinco maneras enseña más que uno que funciona.
2. **Aplicar las cuatro técnicas de regularización** y medir cuánto estrecha cada una el
   hueco entre las dos curvas.
3. **Montar el ciclo de trabajo con retrollamadas**, que es lo que convierte un
   entrenamiento a ciegas en un proceso vigilado.

El caso de trabajo es Fashion-MNIST con un subconjunto pequeño. Es deliberado: **se elige
un conjunto que sobreajusta rápido**, porque para aprender a leer una curva de aprendizaje
hace falta un modelo que falle, no uno bueno.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras

SEMILLA = 20262027
keras.utils.set_random_seed(SEMILLA)

CLASES = ["camiseta", "pantalon", "jersey", "vestido", "abrigo",
          "sandalia", "camisa", "zapatilla", "bolso", "botin"]

(X_todo, y_todo), (X_prueba, y_prueba) = keras.datasets.fashion_mnist.load_data()

# Un subconjunto pequeño, a proposito: sobreajusta deprisa y entrena en segundos.
N = 5000
X_ent, y_ent = X_todo[:N].astype("float32") / 255.0, y_todo[:N]
X_val, y_val = X_todo[N:N + 2000].astype("float32") / 255.0, y_todo[N:N + 2000]
X_pru, y_pru = X_prueba.astype("float32") / 255.0, y_prueba

print(f"entrenamiento {X_ent.shape}   validacion {X_val.shape}   prueba {X_pru.shape}")
print(f"rango de los pixeles: [{X_ent.min():.1f}, {X_ent.max():.1f}]  dtype {X_ent.dtype}")
print(f"perdida inicial esperada: log(10) = {np.log(10):.3f}")

In [ ]:
# Comprobacion de sensatez numero 1: mirar los datos. Siempre.
fig, ejes = plt.subplots(2, 5, figsize=(11, 5))
rng = np.random.default_rng(SEMILLA)
for eje, idx in zip(ejes.ravel(), rng.choice(len(X_ent), 10, replace=False)):
    eje.imshow(X_ent[idx], cmap="gray")
    eje.set_title(CLASES[y_ent[idx]], fontsize=9)
    eje.axis("off")
fig.suptitle("Fashion-MNIST: la etiqueta tiene que corresponder a la imagen", y=1.0)
fig.tight_layout()
plt.show()

---

## 1. Los cinco patrones, fabricados a propósito

Cada uno se produce cambiando **una sola cosa**. Esa es la gracia: el patrón identifica la
causa.

In [ ]:
def red(unidades=(128,), dropout=0.0, l2=0.0):
    reg = keras.regularizers.l2(l2) if l2 else None
    capas = [keras.layers.Input(shape=(28, 28)), keras.layers.Flatten()]
    for u in unidades:
        capas.append(keras.layers.Dense(u, activation="relu", kernel_regularizer=reg))
        if dropout:
            capas.append(keras.layers.Dropout(dropout))
    capas.append(keras.layers.Dense(10, activation="softmax"))
    return keras.Sequential(capas)


def entrena(nombre, unidades=(128,), tasa=1e-3, epocas=40, dropout=0.0, l2=0.0,
            n=N, activacion_salida="softmax", llamadas=None, verbose=0):
    keras.utils.set_random_seed(SEMILLA)
    m = red(unidades, dropout, l2)
    m.compile(optimizer=keras.optimizers.Adam(tasa),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.perf_counter()
    h = m.fit(X_ent[:n], y_ent[:n], epochs=epocas, batch_size=64,
              validation_data=(X_val, y_val), verbose=verbose, callbacks=llamadas or [])
    return m, h, time.perf_counter() - t0


def dibuja(historias, titulo, ylim_perdida=None):
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))
    for nombre, h in historias.items():
        linea, = ejes[0].plot(h.history["loss"], label=f"{nombre} (entrena)")
        ejes[0].plot(h.history["val_loss"], "--", color=linea.get_color(),
                     label=f"{nombre} (valida)")
        linea, = ejes[1].plot(h.history["accuracy"])
        ejes[1].plot(h.history["val_accuracy"], "--", color=linea.get_color())
    ejes[0].set_ylabel("entropia cruzada");  ejes[1].set_ylabel("exactitud")
    for eje in ejes:
        eje.set_xlabel("epoca")
    ejes[0].axhline(np.log(10), color="0.6", lw=1, ls=":",
                    label="log(10): sin aprender")
    ejes[0].legend(fontsize=7)
    if ylim_perdida:
        ejes[0].set_ylim(*ylim_perdida)
    ejes[0].set_title("Perdida: la linea continua entrena, la discontinua valida")
    ejes[1].set_title("Exactitud")
    fig.suptitle(titulo, y=1.02)
    fig.tight_layout()
    plt.show()

In [ ]:
# PATRON 1: SOBREAJUSTE. Muchos parametros, pocos datos, sin precauciones.
_, h_sobre, t = entrena("sobreajuste", unidades=(512, 512), epocas=40, n=2000)
print(f"{t:.1f} s")
dibuja({"512-512 sobre 2000 muestras": h_sobre},
       "Patron 1: SOBREAJUSTE. La de entrenamiento sigue bajando y la de "
       "validacion se da la vuelta")

i = int(np.argmin(h_sobre.history["val_loss"]))
print(f"La validacion toca fondo en la epoca {i} con {h_sobre.history['val_loss'][i]:.3f}")
print(f"y acaba en {h_sobre.history['val_loss'][-1]:.3f}. Todo lo entrenado despues")
print("de esa epoca ha EMPEORADO el modelo. Eso es lo que evita la parada temprana.")

In [ ]:
# PATRON 2: SUBAJUSTE. Un modelo sin capacidad para el problema.
_, h_sub, t = entrena("subajuste", unidades=(2,), epocas=40)
print(f"{t:.1f} s")
dibuja({"una capa de 2 unidades": h_sub},
       "Patron 2: SUBAJUSTE. Las dos curvas altas, planas y pegadas")

print(f"exactitud final: entrena {h_sub.history['accuracy'][-1]:.3f}  "
      f"valida {h_sub.history['val_accuracy'][-1]:.3f}")
print("Las dos igual de malas: el problema NO es que memorice, es que no puede.")
print("Aqui regularizar seria exactamente lo contrario de lo que hace falta.")

In [ ]:
# PATRON 3: TASA DEMASIADO ALTA.
_, h_alta, t = entrena("tasa alta", unidades=(128,), tasa=1.0, epocas=40)
print(f"{t:.1f} s")
dibuja({"Adam con tasa 1.0": h_alta},
       "Patron 3: TASA DEMASIADO ALTA. La perdida oscila o se dispara")

print(f"perdida por epoca: {np.round(h_alta.history['loss'][:8], 3)}")
print(f"hay NaN: {np.isnan(h_alta.history['loss']).any()}")
print(f"log(10) = {np.log(10):.3f}")
print()
print("Mira la secuencia: la primera epoca se dispara a tres cifras y despues")
print("la perdida se queda CLAVADA en log(10), que es no haber aprendido nada.")
print("El paso se pasa tanto del minimo que manda los pesos a una zona muerta")
print("—todas las ReLU apagadas— de la que ya no sale. Con tasas algo menos")
print("extremas lo que se ve es una perdida que oscila en dientes de sierra,")
print("y con algunas combinaciones, NaN directamente.")
print()
print("Los tres sintomas son el mismo problema y se arreglan igual: dividir")
print("la tasa por diez. Y ojo, porque este caso se parece al patron 5 si solo")
print("se mira el final de la curva: hay que mirar la PRIMERA epoca para")
print("distinguirlos.")

In [ ]:
# PATRON 4: TASA DEMASIADO BAJA.
_, h_baja, t = entrena("tasa baja", unidades=(128,), tasa=1e-6, epocas=40)
print(f"{t:.1f} s")
dibuja({"Adam con tasa 1e-6": h_baja},
       "Patron 4: TASA DEMASIADO BAJA. Baja recta y despacio, sin aplanarse")

print(f"perdida en la epoca 0: {h_baja.history['loss'][0]:.3f}")
print(f"perdida en la epoca 39: {h_baja.history['loss'][-1]:.3f}")
print("Va bien, pero a este ritmo harian falta miles de epocas. Se multiplica por diez.")

In [ ]:
# PATRON 5: NO APRENDE. Y este NO es un problema de ajuste: es un ERROR.
# Aqui el error esta puesto a proposito: las etiquetas estan barajadas.
keras.utils.set_random_seed(SEMILLA)
y_roto = np.random.default_rng(SEMILLA).permutation(y_ent[:N])

m_roto = red((128,))
m_roto.compile(optimizer=keras.optimizers.Adam(1e-3),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_roto = m_roto.fit(X_ent[:N], y_roto, epochs=40, batch_size=64,
                    validation_data=(X_val, y_val), verbose=0)

dibuja({"etiquetas barajadas": h_roto},
       "Patron 5: NO APRENDE. La perdida de validacion se queda en log(10)")

print(f"perdida de validacion: de {h_roto.history['val_loss'][0]:.3f} "
      f"a {h_roto.history['val_loss'][-1]:.3f}")
print(f"log(10) = {np.log(10):.3f}")
print()
print("La de ENTRENAMIENTO si baja: la red memoriza etiquetas al azar, que es")
print("justo lo que demuestra que tiene capacidad de sobra. La de validacion")
print("no se mueve porque no hay nada que generalizar.")
print()
print("Este patron no se arregla con hiperparametros. Se arregla mirando los datos,")
print("que es la comprobacion numero 1 del bloque 16.")

### La tabla de diagnóstico

| Patrón | Qué se ve | Qué se hace |
|---|---|---|
| **Sobreajuste** | entrenamiento baja, validación se da la vuelta | parada temprana, dropout, L2, menos capacidad, más datos |
| **Subajuste** | las dos altas, planas y juntas | más capacidad, más épocas, subir la tasa |
| **Tasa alta** | oscila, salta o `NaN` | dividir la tasa por diez |
| **Tasa baja** | baja recta y despacio | multiplicar la tasa por diez |
| **No aprende** | validación clavada en $\log(k)$ | **es un error**: bloque 16 |

> **El quinto no es un problema de entrenamiento.** Distinguirlo de los otros cuatro es la
> diferencia entre arreglarlo en cinco minutos y perder una tarde tocando hiperparámetros.

---

## 2. Regularización, medida

Ahora las cuatro técnicas sobre el mismo caso que sobreajusta, y la pregunta correcta: **no
si baja la pérdida de entrenamiento, sino cuánto se estrecha el hueco**.

In [ ]:
def mide(nombre, **kwargs):
    m, h, t = entrena(nombre, n=2000, epocas=40, **kwargs)
    _, acc_ent = m.evaluate(X_ent[:2000], y_ent[:2000], verbose=0)
    _, acc_val = m.evaluate(X_val, y_val, verbose=0)
    _, acc_pru = m.evaluate(X_pru, y_pru, verbose=0)
    return h, {"tecnica": nombre, "parametros": m.count_params(),
               "exactitud entrena": acc_ent, "exactitud valida": acc_val,
               "exactitud prueba": acc_pru, "hueco": acc_ent - acc_val,
               "epocas": len(h.history["loss"]), "segundos": t}


historias, filas = {}, []
configuraciones = [
    ("sin nada",            dict(unidades=(512, 512))),
    ("menos capacidad",     dict(unidades=(32,))),
    ("dropout 0,5",         dict(unidades=(512, 512), dropout=0.5)),
    ("L2 1e-3",             dict(unidades=(512, 512), l2=1e-3)),
    ("dropout + L2",        dict(unidades=(512, 512), dropout=0.5, l2=1e-4)),
]
for nombre, kwargs in configuraciones:
    h, fila = mide(nombre, **kwargs)
    historias[nombre], _ = h, filas.append(fila)
    print(f"{nombre:20} {fila['segundos']:5.1f} s")

In [ ]:
tabla = pd.DataFrame(filas)
pd.set_option("display.width", 160)
print(tabla.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print("La columna que importa es 'hueco'. Mirar solo 'exactitud entrena' lleva a")
print("la conclusion contraria: sin nada es la que MEJOR entrena, y la peor de todas.")

In [ ]:
dibuja({n: h for n, h in historias.items() if n in
        ("sin nada", "dropout 0,5", "menos capacidad")},
       "Las tres estrategias, sobre el mismo caso", ylim_perdida=(0, 3))

In [ ]:
fig, eje = plt.subplots(figsize=(8, 4.2))
x = np.arange(len(tabla))
eje.bar(x - 0.2, tabla["exactitud entrena"], 0.4, label="entrenamiento")
eje.bar(x + 0.2, tabla["exactitud valida"], 0.4, label="validacion")
for i, hueco in enumerate(tabla["hueco"]):
    eje.annotate(f"{hueco:.3f}", (i, max(tabla['exactitud entrena'][i],
                                         tabla['exactitud valida'][i]) + 0.01),
                 ha="center", fontsize=8)
eje.set_xticks(x, tabla["tecnica"], fontsize=8)
eje.set_ylim(0.5, 1.05)
eje.set_ylabel("exactitud")
eje.set_title("Regularizar es acercar las dos barras, no subir la azul\n"
              "(el numero encima de cada par es el hueco)")
eje.legend(fontsize=8)
fig.tight_layout()
plt.show()

### El orden en que se aplican

No se ponen las cuatro de golpe:

1. **Parada temprana**, siempre. Es gratis, y además ahorra tiempo.
2. **Menos capacidad**, si el modelo es claramente grande para los datos. Quitar una capa
   es más eficaz que añadir dropout a una capa que sobra, y la tabla de arriba lo enseña.
3. **Dropout**, si sigue sobreajustando.
4. **L2**, como ajuste fino.
5. **Más datos o aumento de datos**, que es lo que de verdad funciona cuando se puede.

Y el quinto punto no es retórico. Vamos a medirlo.

In [ ]:
print(f"{'muestras':>9} {'exactitud prueba':>18} {'hueco':>8}")
print("-" * 40)
for n in (500, 1000, 2000, 5000):
    m, h, _ = entrena(f"n={n}", unidades=(512, 512), epocas=25, n=n)
    _, acc_ent = m.evaluate(X_ent[:n], y_ent[:n], verbose=0)
    _, acc_pru = m.evaluate(X_pru, y_pru, verbose=0)
    _, acc_val = m.evaluate(X_val, y_val, verbose=0)
    print(f"{n:>9,} {acc_pru:>18.4f} {acc_ent - acc_val:>8.4f}")
print()
print("Diez veces mas datos, con el MISMO modelo y sin ninguna tecnica de")
print("regularizacion, y el hueco se encoge solo. Por eso 'consigue mas datos'")
print("no es una respuesta perezosa: es la primera respuesta.")

---

## 3. Retrollamadas: el ciclo de trabajo

Una retrollamada es un objeto que Keras invoca en momentos concretos del entrenamiento.
Son el mecanismo con el que un entrenamiento deja de ser una llamada a ciegas.

In [ ]:
llamadas = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint("mejor_modelo.keras", monitor="val_loss",
                                    save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=4, min_lr=1e-6, verbose=1),
]

m_ciclo, h_ciclo, t = entrena("ciclo completo", unidades=(512, 512), epocas=60,
                              n=2000, dropout=0.4, llamadas=llamadas, verbose=0)
print(f"\n{t:.1f} s, {len(h_ciclo.history['loss'])} epocas de las 60 pedidas")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))
ejes[0].plot(h_ciclo.history["loss"], label="entrenamiento")
ejes[0].plot(h_ciclo.history["val_loss"], label="validacion")
mejor = int(np.argmin(h_ciclo.history["val_loss"]))
ejes[0].axvline(mejor, color="crimson", ls="--", lw=1,
                label=f"mejor epoca ({mejor}); los pesos son de aqui")
ejes[0].set_xlabel("epoca");  ejes[0].set_ylabel("entropia cruzada")
ejes[0].set_title("La parada temprana corta, y restore_best_weights\nvuelve a la linea roja")
ejes[0].legend(fontsize=8)

ejes[1].plot(h_ciclo.history["learning_rate"], color="darkgreen")
ejes[1].set_yscale("log")
ejes[1].set_xlabel("epoca");  ejes[1].set_ylabel("tasa de aprendizaje")
ejes[1].set_title("ReduceLROnPlateau la baja a la mitad\ncada vez que se estanca")
fig.tight_layout()
plt.show()

### Por qué se ponen `EarlyStopping` y `ModelCheckpoint` a la vez

Son redundantes si la parada temprana ya restaura los mejores pesos. Se ponen los dos de
todas formas porque el punto de control deja el modelo **en disco**: si el proceso se cae o
la sesión de Colab se desconecta, hay algo que recuperar. Es una precaución barata contra
un accidente caro.

In [ ]:
recuperado = keras.models.load_model("mejor_modelo.keras")
_, acc_memoria = m_ciclo.evaluate(X_pru, y_pru, verbose=0)
_, acc_disco = recuperado.evaluate(X_pru, y_pru, verbose=0)

print(f"modelo en memoria (restore_best_weights): {acc_memoria:.4f}")
print(f"modelo recuperado del disco:              {acc_disco:.4f}")
print()
print("Iguales, porque los dos guardan la misma epoca. La diferencia es que si")
print("el proceso se hubiera caido en la epoca 30, el de disco seguiria estando.")

In [ ]:
# Que restaura exactamente restore_best_weights: la epoca con la mejor
# cantidad VIGILADA. Y esa eleccion tiene consecuencias.
def compara_restauracion(monitor, modo):
    resultados = {}
    for restaurar in (False, True):
        keras.utils.set_random_seed(SEMILLA)
        m, h, _ = entrena("", unidades=(512, 512), epocas=60, n=2000,
                          llamadas=[keras.callbacks.EarlyStopping(
                              monitor=monitor, mode=modo, patience=8,
                              restore_best_weights=restaurar)])
        perdida, exactitud = m.evaluate(X_pru, y_pru, verbose=0)
        resultados[restaurar] = (perdida, exactitud, h)
    return resultados


por_perdida = compara_restauracion("val_loss", "min")

print("Vigilando val_loss:")
print(f"{'':28} {'perdida prueba':>15} {'exactitud prueba':>18}")
for restaurar, (perdida, exactitud, _) in por_perdida.items():
    print(f"  restore_best_weights={str(restaurar):5}      {perdida:>15.4f} "
          f"{exactitud:>18.4f}")

> **Aquí pasa algo que no se esperaba, y es lo más útil de este apartado.** Restaurar los
> mejores pesos mejora la pérdida de prueba, que es lo que se estaba vigilando, y **empeora
> la exactitud de prueba**.
>
> No es un error: es que `val_loss` y la exactitud dejan de ir de la mano. La entropía
> cruzada castiga la confianza equivocada, así que un modelo que sigue entrenando se vuelve
> más seguro de sí mismo —y su pérdida empeora— mientras acierta *más* casos. La época de
> mínima pérdida y la de máxima exactitud no son la misma.
>
> La lección no es que `restore_best_weights` esté mal. Es esta:

**Se vigila la cantidad que se va a informar.** Si la decisión se toma con la exactitud, hay
que vigilar `val_accuracy`; si se toma con el AUC, `val_auc`. Vigilar una cosa y presumir de
otra es una incoherencia silenciosa, y se comprueba en dos líneas:

In [ ]:
por_exactitud = compara_restauracion("val_accuracy", "max")

print("Vigilando val_accuracy:")
print(f"{'':28} {'perdida prueba':>15} {'exactitud prueba':>18}")
for restaurar, (perdida, exactitud, _) in por_exactitud.items():
    print(f"  restore_best_weights={str(restaurar):5}      {perdida:>15.4f} "
          f"{exactitud:>18.4f}")

print()
print("Resumen de las cuatro combinaciones, en exactitud de prueba:")
print(f"{'vigilando':>16} {'sin restaurar':>15} {'restaurando':>13}")
print("-" * 48)
for nombre, r in [("val_loss", por_perdida), ("val_accuracy", por_exactitud)]:
    print(f"{nombre:>16} {r[False][1]:>15.4f} {r[True][1]:>13.4f}")
print()
print("Con el monitor coherente con lo que se informa, restaurar los mejores")
print("pesos gana. Con el monitor incoherente, puede perder. restore_best_weights")
print("no es la decision: la decision es QUE se vigila.")

### Una retrollamada propia

Se escribe heredando de `keras.callbacks.Callback` e implementando el método del momento
que interese. Esta mide el tiempo por época, que es la medición que hace falta en la
**P5.1**.

In [ ]:
class Cronometro(keras.callbacks.Callback):
    # Los metodos disponibles son on_train_begin/end, on_epoch_begin/end,
    # on_batch_begin/end y on_predict_*. Se implementan solo los que hagan falta.
    def on_train_begin(self, logs=None):
        self.tiempos = []

    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.tiempos.append(time.perf_counter() - self._t0)


crono = Cronometro()
_, _, _ = entrena("cronometrado", unidades=(256,), epocas=10, n=2000,
                  llamadas=[crono])

t = np.array(crono.tiempos)
print(f"{len(t)} epocas medidas")
print(f"  primera epoca: {t[0] * 1000:7.1f} ms   <- incluye la construccion del grafo")
print(f"  mediana:       {np.median(t[1:]) * 1000:7.1f} ms")
print(f"  desviacion:    {t[1:].std() * 1000:7.1f} ms")
print()
print("La primera epoca SIEMPRE es mas lenta. Incluirla en una media es el error")
print("de medicion mas comun de esta unidad, y por eso se informa la mediana")
print("de la segunda en adelante.")

---

## 4. Aumento de datos

En imágenes hay una quinta técnica que vale más que las otras cuatro juntas: generar
variantes plausibles de cada imagen. Se pone **dentro del modelo**, como primeras capas:
así solo actúa al entrenar, se desactiva sola al evaluar, y se ejecuta en el acelerador.

In [ ]:
aumento = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.08),
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomTranslation(0.08, 0.08),
], name="aumento")

original = X_ent[7][np.newaxis, ..., np.newaxis]

fig, ejes = plt.subplots(1, 9, figsize=(13, 2))
ejes[0].imshow(original[0, ..., 0], cmap="gray")
ejes[0].set_title("original", fontsize=8)
for eje in ejes:
    eje.axis("off")
for eje in ejes[1:]:
    eje.imshow(aumento(original, training=True)[0, ..., 0], cmap="gray")
fig.suptitle(f"Ocho variantes de la misma prenda ({CLASES[y_ent[7]]}): "
             "todas siguen siendo esa prenda", y=1.12)
fig.tight_layout()
plt.show()

> **La transformación tiene que preservar la etiqueta.** Voltear una camiseta en horizontal
> da una camiseta. Voltear un 6 en vertical da un 9. Voltear una radiografía de tórax
> intercambia el corazón de lado. Aplicar aumento sin pensar en la etiqueta enseña a la red
> cosas falsas, y **no da ningún error**.

Fashion-MNIST tolera el volteo horizontal porque la ropa es aproximadamente simétrica. El
MNIST de dígitos **no**, y conviene comprobarlo antes de copiar el aumento de un cuaderno a
otro.

In [ ]:
keras.utils.set_random_seed(SEMILLA)
con_aumento = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    aumento,
    keras.layers.Flatten(),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
con_aumento.compile(optimizer=keras.optimizers.Adam(1e-3),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_aum = con_aumento.fit(X_ent[:2000, ..., np.newaxis], y_ent[:2000], epochs=40,
                        batch_size=64,
                        validation_data=(X_val[..., np.newaxis], y_val), verbose=0)

_, acc_aum_ent = con_aumento.evaluate(X_ent[:2000, ..., np.newaxis], y_ent[:2000], verbose=0)
_, acc_aum_val = con_aumento.evaluate(X_val[..., np.newaxis], y_val, verbose=0)
sin = tabla[tabla["tecnica"] == "sin nada"].iloc[0]

print(f"{'':18} {'entrena':>9} {'valida':>9} {'hueco':>8}")
print("-" * 48)
print(f"{'sin nada':18} {sin['exactitud entrena']:>9.4f} "
      f"{sin['exactitud valida']:>9.4f} {sin['hueco']:>8.4f}")
print(f"{'con aumento':18} {acc_aum_ent:>9.4f} {acc_aum_val:>9.4f} "
      f"{acc_aum_ent - acc_aum_val:>8.4f}")

---

## Ejercicios

### Ejercicio 1. Reconocer sin mirar el código

Un compañero te pasa cuatro figuras de curvas de aprendizaje sin decirte qué modelo es
cada una. Escribe, para cada patrón de la tabla del apartado 1, **la primera pregunta que
le harías** y **el primer cambio que le propondrías**. Después fabrica tú un quinto caso
que sea ambiguo entre dos patrones, y explica qué medición lo desambigua.

### Ejercicio 2. La paciencia

Entrena el caso que sobreajusta con `patience` en `[2, 5, 10, 20, 50]`. Dibuja la
exactitud de prueba y el número de épocas frente a la paciencia. ¿Hay un valor a partir
del cual la parada temprana deja de servir de nada? Relaciónalo con lo que le pasa a la red
de 4 unidades del cuaderno `UD5_04`.

### Ejercicio 3. Dropout, barrido

Prueba `dropout` en `[0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]` sobre la red 512-512 con 2.000
muestras. Dibuja las tres exactitudes —entrenamiento, validación y prueba— en la misma
figura. ¿Dónde está el óptimo, y qué pasa con 0,9? Escribe una frase sobre por qué
demasiado dropout produce el patrón 2 y no el 1.

### Ejercicio 4. El aumento que estropea

Entrena el mismo modelo sobre **MNIST** (`keras.datasets.mnist`) con
`RandomFlip("horizontal")` y sin él. Compara. Explica el resultado en términos de la regla
de que la transformación tiene que preservar la etiqueta, y di qué dígitos son los que
rompen.

### Ejercicio 5. Retrollamada propia

Escribe una retrollamada que **detenga el entrenamiento si la pérdida se vuelve `NaN`**,
imprimiendo la época y el lote en el que ha pasado. Pruébala provocando el patrón 3.
Compárala con `keras.callbacks.TerminateOnNaN`, que hace lo mismo, y di qué le has añadido.

### Ejercicio 6. El presupuesto

Con el cronómetro del apartado 3, mide el tiempo por época de la red 512-512 con tamaños
de lote 16, 32, 64, 128, 256 y 512. Dibuja tiempo por época y tiempo por paso. ¿Cuál de los
dos es aproximadamente constante, y qué dice eso sobre dónde está el coste? Es la medición
que pide la P5.1.

---

## Lo que hay que llevarse de aquí

1. **Se miran las cuatro curvas antes que cualquier número final.**
2. **Los cinco patrones se producen cambiando una sola cosa**, y por eso el patrón
   identifica la causa.
3. **El quinto patrón no es un problema de ajuste: es un error**, y se arregla mirando los
   datos.
4. **Una pérdida de entrenamiento que baja con etiquetas al azar demuestra que la red tiene
   capacidad de sobra**, y que lo que falta es señal.
5. **Regularizar es acercar las dos curvas, no bajar la de entrenamiento.** Mirar solo el
   entrenamiento lleva a la conclusión contraria.
6. **Quitar capacidad suele ganar a añadir dropout** cuando el modelo es claramente grande.
7. **Más datos estrecha el hueco solo**, sin ninguna técnica. Es la primera respuesta, no
   la perezosa.
8. **Se vigila la cantidad que se va a informar.** `val_loss` y la exactitud dejan de ir
   de la mano en cuanto el modelo se vuelve confiado, y `restore_best_weights` restaura la
   mejor época *de lo que se vigila*, no la mejor a secas.
9. **El punto de control está en disco**, y por eso se pone aunque sea redundante.
10. **La primera época siempre es más lenta**, e incluirla en una media es el error de
    medición más común.
11. **El aumento de datos va dentro del modelo**, y tiene que preservar la etiqueta.